# Diffusion Policy Inference Demo (LIBERO)

Simplified inference notebook for running diffusion policy on LIBERO tasks.

**Prerequisite**: Run a libero environment server `conda activate libero && python scripts/libero_env_server.py --env libero_10`

Run this notebook with `diff_policy` python kernel

## User Configuration

Edit these settings to choose which checkpoint and task to run:

In [43]:
# Path to trained model checkpoint
CHECKPOINT_PATH = (
    "ckpts/exp1_resnet_paraphrase_best/libero_epoch_44_20260418_205512_model.pth"
)

# Output video file path (e.g., "inference_output.mp4")
OUTPUT_VIDEO_PATH = "inf_out.mp4"

# LIBERO task index to run: None = random, list of ints = run those tasks (e.g. [0, 2, 5])
TASK_IDX = [3, 4]

# LIBERO task suite (e.g., "libero_10", "libero_spatial", "libero_goal", "libero_100")
SUITE = "libero_10"

# Optional: override task description. If None, uses server's description
TASK_DESCRIPTION = None

## Setup and Imports

In [44]:
import torch
import numpy as np
import logging
from collections import deque
from tqdm.notebook import tqdm
from diffusers import DDPMScheduler

# Import from repo
from diffusion_policy.model.diffusion_policy import DiffusionPolicy
from diffusion_policy.util.normalization import unnormalize_data, normalize_data
from diffusion_policy.remote_env import RemoteEnv

# Setup logging
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logger.info(f"Using device: {device}")

# ZMQ server address for LIBERO environment
ZMQ_ADDRESS = "tcp://localhost:5555"

[INFO] Using device: cuda


## Helper Functions

In [45]:
def preprocess_state_libero(obs: dict, stats: dict) -> np.ndarray:
    """Normalize LIBERO observation state to match training preprocessing.

    robot0_eef_quat from server is [x,y,z,w] (via T.mat2quat).
    Training uses [w,x,y,z], so we reorder here.
    """
    ee_pos = normalize_data(
        np.asarray(obs["robot0_eef_pos"]).flatten(), stats["obs"]["ee_pos"]
    )
    q = np.asarray(obs["robot0_eef_quat"]).flatten()  # [x,y,z,w]
    ee_quat = np.array([q[3], q[0], q[1], q[2]])  # reorder to [w,x,y,z]
    gripper = normalize_data(
        np.asarray(obs["robot0_gripper_qpos"]).flatten(),
        stats["obs"]["gripper_states"],
    )
    return np.concatenate([ee_pos, ee_quat, gripper])


def denormalize_actions_libero(pred_actions: torch.Tensor, stats: dict) -> np.ndarray:
    """Convert 10D model output back to 7D LIBERO env actions.

    Model outputs: [pos_norm(3), rot_6d(6), gripper_norm(1)]
    Env expects:   [pos(3), axis_angle(3), gripper(1)]
    """
    import diffusion_policy.util.rotation as RotationUtils

    pos_np = unnormalize_data(pred_actions[..., :3].numpy(), stats["actions"]["ee_pos"])
    gripper_np = unnormalize_data(
        pred_actions[..., 9:].numpy(), stats["actions"]["gripper_states"]
    )

    rot_6d = pred_actions[..., 3:9]
    rot_matrix = RotationUtils.rotation_6d_to_matrix(rot_6d)
    rot_aa_np = RotationUtils.matrix_to_axis_angle(rot_matrix).numpy()

    return np.concatenate([pos_np, rot_aa_np, gripper_np], axis=-1)

## Load Model

In [46]:
logger.info(f"Loading checkpoint from {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

model_config = checkpoint["model_config"]
state_dict = checkpoint["model_state_dict"]
logger.info(f"Loaded model config from checkpoint")

# Load model
diff_model = DiffusionPolicy(**model_config).to(device)
diff_model.load_state_dict(state_dict)
diff_model.eval()
logger.info(f"Model loaded successfully")

[INFO] Loading checkpoint from ckpts/exp1_resnet_paraphrase_best/libero_epoch_44_20260418_205512_model.pth
[INFO] Loaded model config from checkpoint
[INFO] HTTP Request: HEAD https://huggingface.co/google/siglip2-base-patch16-224/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/siglip2-base-patch16-224/75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2/config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: HEAD https://huggingface.co/google/siglip2-base-patch16-224/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

SiglipTextModel LOAD REPORT from: google/siglip2-base-patch16-224
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.head.attention.in_proj_bias                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.head.mlp.fc1.weight                               | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
logit_bias                      

Vision encoder: resnet18, Text encoder: siglip2-base-patch16-224
Condition features dimension: 1426
Number of Parameters: 578,715,082


[INFO] Model loaded successfully


## Load Normalization Stats

In [47]:
stats = checkpoint["data_stats"]
logger.info("Loaded normalization stats from checkpoint")
logger.info(f"Stats keys: {list(stats.keys())}")

[INFO] Loaded normalization stats from checkpoint
[INFO] Stats keys: ['obs', 'actions']


## Setup Environment and Noise Scheduler

In [48]:
# Close any existing env from a previous run
try:
    env.close()
except Exception:
    pass

# Connect to LIBERO environment server
logger.info(f"Connecting to LIBERO env at {ZMQ_ADDRESS}")
env = RemoteEnv(address=ZMQ_ADDRESS)
env.reset(suite_name=SUITE)

# Get available tasks
available_tasks = env.get_tasks()
logger.info(f"Found {len(available_tasks)} tasks")

# Build list of tasks to run
if TASK_IDX is None:
    import random

    tasks_to_run = [random.choice(available_tasks)]
else:
    tasks_to_run = [
        next((t for t in available_tasks if t["idx"] == idx), available_tasks[idx])
        for idx in TASK_IDX
    ]

for t in tasks_to_run:
    logger.info(f"  Task {t['idx']}: {t['description']}")

# Setup noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=100,
    beta_schedule="squaredcos_cap_v2",
    clip_sample=True,
    prediction_type="epsilon",
)

[INFO] Connecting to LIBERO env at tcp://localhost:5555
[INFO] Found 10 tasks
[INFO]   Task 3: put the black bowl in the bottom drawer of the cabinet and close it
[INFO]   Task 4: put the white mug on the left plate and put the yellow and white mug on the right plate


## Run Inference

In [ ]:
# Get model config values
obs_horizon = model_config.get("obs_horizon", 2)
action_pred_horizon = model_config.get("action_pred_horizon", 16)
action_exec_horizon = 8  # Fixed for LIBERO
max_steps = 300

import imageio

all_video_frames = []
all_results = []

# Derive per-task video path: e.g. "inf_out.mp4" -> "inf_out_task3.mp4"
_base, _ext = OUTPUT_VIDEO_PATH.rsplit(".", 1)

for task in tasks_to_run:
    task_idx = task["idx"]
    task_description = TASK_DESCRIPTION or task["description"]
    logger.info(f"\n--- Task {task_idx}: {task_description} ---")

    # Reset environment to selected task
    obs = env.reset(task_idx=task_idx, suite_name=SUITE)

    # Initialize observation buffers
    obs_images = deque(maxlen=obs_horizon)
    obs_states = deque(maxlen=obs_horizon)

    img = obs["agentview_image"]
    state = preprocess_state_libero(obs, stats)
    for _ in range(obs_horizon):
        obs_images.append(img)
        obs_states.append(state)

    logger.info(f"Initial observation shape - image: {img.shape}, state: {state.shape}")

    rewards = []
    video_frames = []
    step_idx = 0
    done = False

    logger.info(f"Starting inference loop (max {max_steps} steps)...")

    with tqdm(total=max_steps, desc=f"Task {task_idx}", unit="step") as pbar:
        while not done:
            # === Build observation tensors ===
            images_np = np.stack(obs_images)  # (obs_h, H, W, 3)
            images_np = np.moveaxis(images_np, -1, 1)  # (obs_h, 3, H, W)
            images_np = images_np / 255.0
            images = torch.from_numpy(images_np).float().unsqueeze(0).to(device)

            states_np = np.stack(obs_states)  # (obs_h, state_dim)
            states = torch.from_numpy(states_np).float().unsqueeze(0).to(device)

            # === DDPM denoising loop ===
            noisy_actions = torch.randn(
                (1, action_pred_horizon, model_config["action_dim"]),
                device=device,
            )
            noise_scheduler.set_timesteps(100)

            task_desc = (
                [task_description]
                if task_description is not None and diff_model.lang_encoder is not None
                else None
            )

            with torch.no_grad():
                for t in noise_scheduler.timesteps:
                    noise_pred = diff_model(
                        noisy_actions,
                        torch.full((1,), t, device=device, dtype=torch.long),
                        images,
                        state_obs_seq=states,
                        task_description=task_desc,
                    )
                    noisy_actions = noise_scheduler.step(
                        noise_pred, t, noisy_actions
                    ).prev_sample

            # === Denormalize predicted actions ===
            pred_actions = denormalize_actions_libero(
                noisy_actions.detach().cpu()[0], stats
            )

            start = obs_horizon - 1
            end = start + action_exec_horizon
            action = pred_actions[start:end, :]

            # === Execute actions ===
            for i in range(len(action)):
                obs, reward, done, info = env.step(action[i])
                rewards.append(reward)

                obs_images.append(obs["agentview_image"])
                obs_states.append(preprocess_state_libero(obs, stats))

                render_img = env.render()
                if render_img is not None:
                    video_frames.append(render_img)

                step_idx += 1
                pbar.update(1)
                pbar.set_postfix(reward=f"{reward:.3f}")

                if step_idx > max_steps:
                    done = True

                if done:
                    break

    # Save per-task video immediately after it finishes (flip vertically)
    if video_frames:
        task_video_path = f"{_base}_task{task_idx}.{_ext}"
        imageio.mimwrite(task_video_path, [f[::-1] for f in video_frames], fps=15)
        logger.info(f"Saved video ({len(video_frames)} frames) to {task_video_path}")

    all_video_frames.extend(video_frames)
    all_results.append(
        {
            "task_idx": task_idx,
            "description": task_description,
            "steps": step_idx,
            "total_reward": sum(rewards),
            "max_reward": max(rewards) if rewards else 0,
            "mean_reward": sum(rewards) / len(rewards) if rewards else 0,
        }
    )
    logger.info(f"Task {task_idx} done — steps: {step_idx}, reward: {sum(rewards):.2f}")

logger.info(f"\nAll tasks complete. Total frames: {len(all_video_frames)}")

[INFO] 
--- Task 3: put the black bowl in the bottom drawer of the cabinet and close it ---
[INFO] Initial observation shape - image: (128, 128, 3), state: (9,)
[INFO] Starting inference loop (max 300 steps)...


Task 3:   0%|          | 0/300 [00:00<?, ?step/s]

[INFO] Saved video (290 frames) to inf_out_task3.mp4
[INFO] Task 3 done — steps: 290, reward: 1.00
[INFO] 
--- Task 4: put the white mug on the left plate and put the yellow and white mug on the right plate ---
[INFO] Initial observation shape - image: (128, 128, 3), state: (9,)
[INFO] Starting inference loop (max 300 steps)...


Task 4:   0%|          | 0/300 [00:00<?, ?step/s]

[INFO] Saved video (301 frames) to inf_out_task4.mp4
[INFO] Task 4 done — steps: 301, reward: 0.00
[INFO] 
All tasks complete. Total frames: 591


In [50]:
video_file = "inf_out_task3.mp4"
HTML(f"""
    <video width="640" height="480" controls>
        <source src="{video_file}" type="video/mp4">
    </video>
    <script>
        var video = document.getElementsByTagName('video')[0];
        video.playbackRate = 2.0; // Increase the playback speed to 2x
        </script>    
""")

In [51]:
video_file = "inf_out_task4.mp4"
HTML(f"""
    <video width="640" height="480" controls>
        <source src="{video_file}" type="video/mp4">
    </video>
    <script>
        var video = document.getElementsByTagName('video')[0];
        video.playbackRate = 2.0; // Increase the playback speed to 2x
        </script>    
""")